In [36]:
################ Import necessary libraries
import pandas as pd
import numpy as np

In [37]:
################ Import processed data
market_share = pd.read_excel("C:/Users/Lenovo/Desktop/Dissertaion/Code/MSc-Dissertation/final_cleaned_data.xlsx")

# Import the steel price data
steel_price_file = pd.read_excel(
    "C:/Users/Lenovo/Desktop/Dissertaion/China Data/Instruments/steel price index.xlsx",
    sheet_name='Sheet1')

# Convert the steel price data to a DataFrame
steel_price = pd.DataFrame(steel_price_file)

In [ ]:
################ Construct differentiation IVs
# Key idea is to express the position of a brand in product characteristic space

# Define funciton to construct differentiation IVs

def construct_differentiation_ivs(market_share, threshold_std=1.0):
    """
    Construct IVs using positional indexing (6th col as mass, 7th as power)
    
    Parameters:
    -----------
    market_share : DataFrame
        Must have at least 7 columns with:
        - 6th column as mass
        - 7th column as power
    threshold_std : float
        Number of standard deviations for local difference threshold
    
    Returns:
    --------
    DataFrame with original data plus 6 new IV columns
    """
    
    # Create working copy
    df = market_share.copy()
    
    # Get columns by position
    mass_col = df.columns[6]
    power_col = df.columns[7]
    
    print(f"Using columns: {mass_col} as mass, {power_col} as power")
    
    # Calculate IVs
    
    ## Sum of rivals
    df['sum_rival_mass'] = df.groupby('market_id')[mass_col].transform('sum') - df[mass_col]
    df['sum_rival_power'] = df.groupby('market_id')[power_col].transform('sum') - df[power_col]
    
    ## Euclidean distance (vectorized)
    def euclidean(group, col):
        vals = group[col].values.astype(np.float32)
        diffs = np.subtract.outer(vals, vals)
        np.fill_diagonal(diffs, 0)
        return np.square(diffs).sum(axis=1)
    
    df['euclidean_mass'] = df.groupby('market_id').apply(lambda x: euclidean(x, mass_col)).explode().values
    df['euclidean_power'] = df.groupby('market_id').apply(lambda x: euclidean(x, power_col)).explode().values
    
    ## Local difference
    mass_thresh = threshold_std * df[mass_col].std()
    power_thresh = threshold_std * df[power_col].std()
    
    def local_diff(group, col, threshold):
        vals = group[col].values.astype(np.float32)
        diffs = np.abs(np.subtract.outer(vals, vals))
        np.fill_diagonal(diffs, 0)
        return (diffs < threshold).sum(axis=1)
    
    df['local_mass'] = df.groupby('market_id').apply(lambda x: local_diff(x, mass_col, mass_thresh)).explode().values
    df['local_power'] = df.groupby('market_id').apply(lambda x: local_diff(x, power_col, power_thresh)).explode().values
    
    return df


# Construct IVs
market_share_with_ivs = construct_differentiation_ivs(market_share, threshold_std=1)

# Display results
print(market_share_with_ivs)

# Save to CSV
market_share_with_ivs.to_csv('market_share_with_ivs.csv', index=False)
print("\nResults saved to 'market_share_with_ivs.csv'")

In [ ]:
################ Construct exogenous cost-shifters

# Convert Year column into year in steel price data
steel_price['year'] = pd.to_datetime(steel_price['Year']).dt.year
# Drop 'Year' column as it's no longer needed
steel_price.drop(columns=['Year'], inplace=True)

# Rename columns for clarity
steel_price.rename(columns={'China: Steel Composite Price Indices:Annual:Average': 'steel_price_index'}, inplace=True)

# Use the mass of a model interacted with the steel price index as an exogenous cost-shifter
def construct_cost_shifters(market_share, steel_price):
    """
    Construct exogenous cost-shifters using weight and steel price index.
    
    Parameters:
    -----------
    market_share : DataFrame
        Must have a 'mass' column
    steel_price : DataFrame
        Must have a 'steel_price_index' column
    
    Returns:
    --------
    DataFrame with original data plus cost-shifter column
    """
    
    # Ensure steel price is aligned with market share data
    steel_price = steel_price.set_index('year')  # Assuming 'date' is the index
    
    # Merge on date (assuming both DataFrames have a 'date' column)
    merged = market_share.merge(steel_price, on='date', how='left')
    
    # Create cost-shifter as weight * steel price index
    merged['cost_shifter'] = merged['weight'] * merged['steel_price_index']
    
    return merged

# Construct cost-shifters
market_share_with_ivs_2 = construct_cost_shifters(market_share_with_ivs, steel_price)

In [ ]:
################ Construct charging station IVs

# Shift-share Instrument: interacting national stock with local number of 

Index(['year', 'type', 'province', 'brand', 'model', 'fuel_type', 'mass',
       'power', 'sales', 'weighted_Avg_Price', 'market_size', 'market_share',
       'province_id', 'brand_id', 'model_id', 'product_id', 'market_id',
       'charging_stations_stock', 'sum_rival_mass', 'sum_rival_power',
       'euclidean_mass', 'euclidean_power', 'local_mass', 'local_power',
       'steel_price_index', 'cost_shifter'],
      dtype='object')